In [ ]:
# ============================================
# Train on train.csv → Validate (tqdm sMAPE) → Predict on TEST_*.csv
# NLinear + BiLSTM (3 seeds)  ⟶ per-key family 가중 + seed 가중
# Features (exactly as requested, 20개):
# ['store_name_id','menu_name_id','season_id','peak_season_id','dow','is_open','sales_quartile',
#  'lag_1','lag_7','lag_14','lag_28','roll_mean_7','roll_std_7','roll_mean_28',
#  'zero_run_len','is_zero_run','days_since_nonzero','pct_zero_7','pct_zero_14','sales_log1p']
# Output:
#   BASE_PATH\submission.csv (long: 영업일자, 영업장명_메뉴명, 매출수량)
#   BASE_PATH\submission_wide.csv (sample_submission order)
#   BASE_PATH\validation_overall.csv / validation_per_horizon.csv / validation_by_key.csv
# ============================================



import os, re, glob, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from contextlib import nullcontext

import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

# ----------------
# Paths
# ----------------
BASE_PATH   = r"C:\Users\tjdql\OneDrive\바탕 화면\open(2)"
TRAIN_PATH  = os.path.join(BASE_PATH, "train", "train.csv")
TEST_GLOB   = os.path.join(BASE_PATH, "test", "TEST_*.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_PATH, "sample_submission.csv")
FAMILY_W_PATHS  = [
    os.path.join(BASE_PATH, "family_weights.csv"),
    "/mnt/data/family_weights.csv",  # 업로드 폴더 fallback
]

# ----------------
# Config
# ----------------
class CFG:
    LOOKBACK = 28
    PREDICT  = 7
    BATCH    = 32
    EPOCHS   = 60
    EPOCHS_SPECIAL = 45
    LR       = 1e-3
    ZERO_IGNORE = True

    # ✅ 3-seed 앙상블 (seed 가중은 아래 get_seed_weights()에서 자동 정규화)
    SEEDS = [25, 42, 2025]
    SEED_WEIGHTS = None  # 예: [0.5, 0.3, 0.2]로 바꿔도 자동 정규화됨

    KEY_COL = "영업장명_메뉴명"
    FEATS = None  # 실행 중 셋업

    SPECIAL_PREFIX = []      # 특수 매장 없음(요청대로)
    TREND_CLIP = (0.9, 1.1)  # (미사용)

# family 기본 가중(없으면 0.5/0.5)
W_DEFAULT = (0.5, 0.5)   # (BiLSTM, NLinear)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass
print("Device:", DEVICE)

# ----------------
# Utils
# ----------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def smape_np(y_true, y_pred, zero_ignore=True):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    if zero_ignore:
        mask = (y_true != 0)
        if mask.sum() == 0: return np.nan
        y_true, y_pred = y_true[mask], y_pred[mask]
    den = np.abs(y_true) + np.abs(y_pred)
    den = np.where(den == 0, 1e-6, den)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / den))

class SMAPELossTorch(nn.Module):
    def __init__(self, eps=1e-6, zero_ignore=True):
        super().__init__(); self.eps = eps; self.zero_ignore = zero_ignore
    def forward(self, y_pred, y_true):
        if self.zero_ignore:
            mask = (torch.abs(y_true) > 0)
            if not mask.any():
                return y_pred.sum() * 0.0
            y_pred = y_pred[mask]; y_true = y_true[mask]
        num = torch.abs(y_pred - y_true)
        den = torch.clamp(torch.abs(y_pred) + torch.abs(y_true), min=self.eps)
        return torch.mean(2.0 * num / den)

def get_store_name(key: str) -> str:
    return str(key).split('_')[0]

def is_special_store(store: str) -> bool:
    return any(store.startswith(p) for p in CFG.SPECIAL_PREFIX)

def _norm_pair(wb, wn):
    wb = max(0.0, float(wb)); wn = max(0.0, float(wn))
    s = wb + wn
    if s <= 0: return W_DEFAULT
    return (wb/s, wn/s)

# ===== Seed 가중 (자동 정규화) =====
def get_seed_weights():
    if (CFG.SEED_WEIGHTS is None) or (len(CFG.SEED_WEIGHTS) != len(CFG.SEEDS)):
        w = np.ones(len(CFG.SEEDS), dtype=float)
    else:
        w = np.array(CFG.SEED_WEIGHTS, dtype=float)
    s = np.clip(w.sum(), 1e-8, None)
    return (w / s).tolist()

# ===== Family 가중 CSV 로딩 =====
FAMILY_WEIGHT_MAP = {}  # key -> (wb, wn)

def load_family_weight_csv():
    global FAMILY_WEIGHT_MAP
    for p in FAMILY_W_PATHS:
        if os.path.isfile(p):
            try:
                df = pd.read_csv(p)
                cols = {c.lower(): c for c in df.columns}
                # key 컬럼 추론
                key_col = None
                for cand in ["영업장명_메뉴명", "key", "store_item", "item", "menu_key"]:
                    if cand.lower() in cols:
                        key_col = cols[cand.lower()]; break
                if key_col is None:
                    # 첫 컬럼을 키로 가정
                    key_col = df.columns[0]

                # bi/nn 컬럼 추론
                bi_col = None; nl_col = None
                for c in df.columns:
                    cl = c.lower()
                    if any(x in cl for x in ["bilstm","bi","wb"]) and bi_col is None:
                        bi_col = c
                    if any(x in cl for x in ["nlinear","nl","wn"]) and nl_col is None:
                        nl_col = c
                # 안전장치
                if bi_col is None or nl_col is None:
                    # 흔한 조합 시도
                    if "w_bilstm" in df.columns and "w_nlinear" in df.columns:
                        bi_col, nl_col = "w_bilstm", "w_nlinear"
                    elif "wb" in df.columns and "wn" in df.columns:
                        bi_col, nl_col = "wb", "wn"

                if bi_col is None or nl_col is None:
                    print(f"⚠️ family_weights.csv 컬럼 해석 실패: {p} (사용 안함)")
                    return

                cnt = 0
                for _, r in df.iterrows():
                    k = str(r[key_col])
                    wb, wn = r[bi_col], r[nl_col]
                    FAMILY_WEIGHT_MAP[k] = _norm_pair(wb, wn)
                    cnt += 1
                print(f"✅ Family 가중 로드: {cnt} keys from {p}")
                return
            except Exception as e:
                print(f"⚠️ family_weights.csv 읽기 실패({p}):", e)
                return
    print("ℹ️ family_weights.csv 미발견 → 기본 가중(0.5/0.5) 사용")

def get_family_weights_for_key(key: str):
    # 정확히 매칭되면 사용
    if key in FAMILY_WEIGHT_MAP:
        return FAMILY_WEIGHT_MAP[key]
    # 못 찾으면 기본
    return W_DEFAULT

# ----------------
# 0) 데이터 적응적 로더 (train/test 어떤 포맷이 와도 영어 표준으로 변환)
# ----------------
def load_any_csv_to_en(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    cols = set(d.columns)

    # KR → EN
    if '영업일자' in cols:
        d['date'] = pd.to_datetime(d['영업일자'])
        if '영업장명_메뉴명' in cols:
            tmp = d['영업장명_메뉴명'].astype(str).str.split('_', n=1, expand=True)
            d['store_name'] = tmp[0].astype(str)
            d['menu_name']  = tmp[1].astype(str)
        else:
            if '영업장명' in cols:
                d['store_name'] = d['영업장명'].astype(str)
            elif '영업장' in cols:
                d['store_name'] = d['영업장'].astype(str)
            else:
                comp = d.iloc[:,1].astype(str)
                tmp = comp.str.split('_', n=1, expand=True)
                d['store_name'] = tmp[0]; d['menu_name'] = tmp[1]
            if '메뉴명' in cols:
                d['menu_name'] = d.get('menu_name', d['메뉴명']).astype(str)
        if '매출수량' in cols:
            d['sales'] = pd.to_numeric(d['매출수량'], errors='coerce')
        elif 'quant' in cols:
            d['sales'] = pd.to_numeric(d['quant'], errors='coerce')
        else:
            value_col = [c for c in d.columns if c not in ['date','store_name','menu_name']][0]
            d['sales'] = pd.to_numeric(d[value_col], errors='coerce')
    else:
        if 'date' not in cols:
            d = d.rename(columns={d.columns[0]:'date'})
        d['date'] = pd.to_datetime(d['date'], errors='coerce')

        if 'store_name' not in cols or 'menu_name' not in cols:
            if 'store_item' in cols:
                tmp = d['store_item'].astype(str).str.split('_', n=1, expand=True)
                d['store_name'] = tmp[0]; d['menu_name'] = tmp[1]
            else:
                d = d.rename(columns={d.columns[1]:'store_name', d.columns[2]:'menu_name'})
        if 'sales' not in cols:
            val_cand = [c for c in d.columns if c not in ['date','store_name','menu_name']][0]
            d['sales'] = pd.to_numeric(d[val_cand], errors='coerce')

    out = d[['date','store_name','menu_name','sales']].copy()
    out['sales'] = out['sales'].fillna(0).clip(lower=0)
    return out

# ----------------
# 1) 스케줄/시즌/라벨/시계열 피처 생성 (EN df → 확장 피처)
# ----------------
STORE_SCHEDULE = {
    "담하":       [0,1,2,3,4,5,6],
    "미라시아":   [0,1,2,3,4,5,6],
    "라그로타":   [1,2,3,4,5,6],
    "카페테리아": [0,1,2,3,4,5,6],
    "느티나무 셀프BBQ":[1,2,3,4,5,6],
    "연회장":     [1,2,3,4,5,6],
    "포레스트릿": [5,6],
    "화담숲주막": [1,2,3,4,5,6],
    "화담숲카페":[1,2,3,4,5,6],
}  # 증명할수잇는 자료 시각화해서 -> 데이터를 보면 뭐 월요일에는 안연다.

def add_schedule_and_calendar(df_en: pd.DataFrame) -> pd.DataFrame:
    df = df_en.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['dow'] = df['date'].dt.dayofweek

    rows=[]
    for store, days in STORE_SCHEDULE.items():
        for d in days:
            rows.append({'store_name':store, 'dow':d, 'is_open':1})
    sched = pd.DataFrame(rows)
    df = df.merge(sched, on=['store_name','dow'], how='left')
    df['is_open'] = df['is_open'].fillna(0).astype(int)

    def get_season(m):
        if m in [3,4,5]: return "Spring"
        if m in [6,7,8]: return "Summer"
        if m in [9,10,11]: return "Fall"
        return "Winter"
    def get_peak(m):
        return "Peak" if m in [6,7,8,12,1,2] else "Off"
    df['season'] = df['date'].dt.month.map(get_season)
    df['peak_season'] = df['date'].dt.month.map(get_peak)
    return df

def fit_label_maps(train_en: pd.DataFrame):
    maps = {}
    for c in ['store_name','menu_name','season','peak_season']:
        cats = pd.Categorical(train_en[c].astype(str)).categories
        maps[c] = {cat:i for i,cat in enumerate(cats)}
    return maps

def apply_label_maps(df: pd.DataFrame, maps: dict) -> pd.DataFrame:
    out = df.copy()
    def enc(col, name):
        m = maps.get(name, {})
        return out[col].astype(str).map(m).fillna(-1).astype(int)
    out['store_name_id'] = enc('store_name','store_name')
    out['menu_name_id']  = enc('menu_name','menu_name')
    out['season_id']     = enc('season','season')
    out['peak_season_id']= enc('peak_season','peak_season')
    return out

def compute_menu_quartile_map(train_en: pd.DataFrame) -> dict:
    grp = train_en.groupby('menu_name')['sales'].sum().reset_index()
    grp['sales_quartile'] = pd.qcut(grp['sales'], q=4, labels=[1,2,3,4])
    return dict(zip(grp['menu_name'].astype(str), grp['sales_quartile'].astype(int)))

def apply_sales_quartile_map(df_any: pd.DataFrame, quart_map: dict) -> pd.DataFrame:
    out = df_any.copy()
    out['sales_quartile'] = out['menu_name'].astype(str).map(quart_map).fillna(1).astype(int)
    return out

def add_ts_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.sort_values(['store_name','menu_name','date']).copy()
    g = d.groupby(['store_name','menu_name'], sort=False)

    # lags
    for lag in [1,7,14,28]:
        d[f'lag_{lag}'] = g['sales'].shift(lag)

    # rolling (shift(1) to avoid leakage)
    d['roll_mean_7']  = g['sales'].transform(lambda s: s.shift(1).rolling(7,  min_periods=1).mean())
    d['roll_std_7']   = g['sales'].transform(lambda s: s.shift(1).rolling(7,  min_periods=2).std())
    d['roll_mean_28'] = g['sales'].transform(lambda s: s.shift(1).rolling(28, min_periods=1).mean())
    d['roll_std_7']   = d['roll_std_7'].fillna(0)

    # zero-run
    def zero_run_len(s):
        run=0; out=[]
        for v in s:
            run = run+1 if v==0 else 0
            out.append(run)
        return pd.Series(out, index=s.index)
    def days_since_nonzero(s):
        d0=0; out=[]
        for v in s:
            d0 = 0 if v!=0 else d0+1
            out.append(d0)
        return pd.Series(out, index=s.index)
    d['zero_run_len']       = g['sales'].apply(zero_run_len).reset_index(level=[0,1], drop=True)
    d['is_zero_run']        = (d['zero_run_len'] > 0).astype(int)
    d['days_since_nonzero'] = g['sales'].apply(days_since_nonzero).reset_index(level=[0,1], drop=True)

    for w in [7,14]:
        d[f'pct_zero_{w}'] = g['sales'].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).apply(lambda x: (x==0).mean(), raw=True)
        )

    d['sales_log1p'] = np.log1p(d['sales'])
    return d

def smooth_outliers_percentile_kr(kr_df: pd.DataFrame, low_q=0.01, high_q=0.99) -> pd.DataFrame:
    """KR 스펙(영업일자/영업장명_메뉴명/매출수량)에서 1~99퍼센타일 밖은 시간보간"""
    df = kr_df.sort_values([CFG.KEY_COL, '영업일자']).copy()
    def _proc(g):
        s = g['매출수량'].astype(float).copy()
        if s.notna().sum() < 5:
            g['매출수량'] = s.clip(lower=0).round(); return g
        lo, hi = s.quantile(low_q), s.quantile(high_q)
        s[(s < lo) | (s > hi)] = np.nan
        s.index = pd.DatetimeIndex(g['영업일자'])
        s = s.sort_index().interpolate(method='time').ffill().bfill()
        s = s.reindex(pd.DatetimeIndex(g['영업일자']))
        g['매출수량'] = s.values.clip(min=0).round()
        return g
    return df.groupby(CFG.KEY_COL, group_keys=False).apply(_proc)

def finalize_kr_features(kr: pd.DataFrame) -> pd.DataFrame:
    kr = kr.sort_values([CFG.KEY_COL, '영업일자']).copy()
    feat_cols = [c for c in kr.columns if c not in ['영업일자', CFG.KEY_COL]]
    kr[feat_cols] = kr.groupby(CFG.KEY_COL, group_keys=False)[feat_cols].apply(lambda g: g.ffill().bfill())
    kr[feat_cols] = kr[feat_cols].fillna(0)
    for c in feat_cols:
        kr[c] = pd.to_numeric(kr[c], errors='coerce').fillna(0)
    return kr

# ----------------
# 2) KR 스펙 변환 + 요청 피처만 남기기
# ----------------
REQ_FEATURES = [
    'store_name_id','menu_name_id','season_id','peak_season_id',
    'dow','is_open','sales_quartile',
    'lag_1','lag_7','lag_14','lag_28',
    'roll_mean_7','roll_std_7','roll_mean_28',
    'zero_run_len','is_zero_run','days_since_nonzero','pct_zero_7','pct_zero_14',
    'sales_log1p'
]

def en_to_kr_with_feats(df_en: pd.DataFrame) -> pd.DataFrame:
    out = df_en.copy()
    must = {"date","store_name","menu_name","sales"}
    missing = must - set(out.columns)
    assert not missing, f"필수 컬럼 누락: {missing}"

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["영업일자"] = out["date"]
    out["영업장명_메뉴명"] = out["store_name"].astype(str) + "_" + out["menu_name"].astype(str)
    out["매출수량"] = pd.to_numeric(out["sales"], errors="coerce")

    keep = ["영업일자","영업장명_메뉴명","매출수량"] + [c for c in REQ_FEATURES if c in out.columns]
    # 누락 피처 알림
    missing_feats = [c for c in REQ_FEATURES if c not in out.columns]
    if missing_feats:
        print("⚠️ 누락된 피처(0으로 채움 예정):", missing_feats)
        for c in missing_feats:
            out[c] = np.nan

    kr = out[keep].sort_values(["영업장명_메뉴명","영업일자"]).reset_index(drop=True)
    return kr

# ----------------
# 3) 모델
# ----------------
class NLinear(nn.Module):
    def __init__(self, input_len=28, output_len=7, input_dim=5):
        super().__init__()
        self.linear = nn.Linear(input_len, output_len)
    def forward(self, x):                # x: [B, L, D]
        x = x.permute(0,2,1)             # [B, D, L]
        x = self.linear(x)               # [B, D, H]
        return x.permute(0,2,1)[:, :, 0] # [B, H]

class BiLSTM(nn.Module):
    def __init__(self, input_dim=5, hidden=64, num_layers=1, output_len=7, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden,
                            num_layers=num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers>1 else 0.0)
        self.fc = nn.Linear(hidden*2, output_len)
    def forward(self, x):                # x: [B, L, D]
        out, (hn, cn) = self.lstm(x)     # hn: [2, B, H]
        h_last = torch.cat((hn[-2], hn[-1]), dim=1)  # [B, 2H]
        return self.fc(h_last)           # [B, H]

def build_model(name: str, input_len, output_len, input_dim):
    if name == "NLinear":
        return NLinear(input_len, output_len, input_dim)
    elif name == "BiLSTM":
        return BiLSTM(input_dim=input_dim, hidden=64, num_layers=1, output_len=output_len)
    else:
        raise ValueError(name)

# ----------------
# 4) 텐서 변환/학습/예측
# ----------------
def make_tensor_dataset(df, feature_cols, lookback, predict):
    vals = df[feature_cols].values.astype(float)
    good = np.isfinite(vals).all(axis=1)
    vals = vals[good]
    if len(vals) < lookback + predict:
        return None, None
    X, y = [], []
    for i in range(len(vals) - lookback - predict + 1):
        X.append(vals[i:i+lookback])
        y.append(vals[i+lookback:i+lookback+predict, 0])  # 첫 열 = 매출수량
    X = torch.tensor(np.array(X)).float().to(DEVICE)
    y = torch.tensor(np.array(y)).float().to(DEVICE)
    return X, y

def train_tensor_model(X, y, model_name, epochs, lr=CFG.LR, zero_ignore=CFG.ZERO_IGNORE):
    model = build_model(model_name, CFG.LOOKBACK, CFG.PREDICT, len(CFG.FEATS)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = SMAPELossTorch(zero_ignore=zero_ignore)

    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    amp_ctx = torch.cuda.amp.autocast(dtype=torch.float16, enabled=use_amp) if use_amp else nullcontext()

    model.train()
    for _ in range(epochs):
        idx = torch.randperm(len(X), device=X.device)
        for i in range(0, len(X), CFG.BATCH):
            b = idx[i:i+CFG.BATCH]
            xb, yb = X[b].to(DEVICE, non_blocking=True), y[b].to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with amp_ctx:
                pred = model(xb)
                loss = criterion(pred, yb)

            if use_amp:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
    model.eval()
    return model

def predict_next7_from_recent(df_recent, feature_cols, scaler, model):
    df_recent = df_recent.sort_values('영업일자')
    arr = df_recent[feature_cols].values.astype(float)
    if len(arr) < CFG.LOOKBACK: return None
    recent = arr[-CFG.LOOKBACK:]
    X = torch.tensor(scaler.transform(recent)[None, :, :]).float().to(DEVICE)
    with torch.no_grad():
        pred_scaled = model(X).squeeze().cpu().numpy()  # (7,)
    out = []
    for i in range(CFG.PREDICT):
        dummy = np.zeros((1, len(feature_cols)), dtype=float)
        dummy[0,0] = pred_scaled[i]   # 첫 열(매출수량)을 역변환
        out.append(float(max(0.0, scaler.inverse_transform(dummy)[0,0])))
    return out

# ----------------
# 5) 학습 캐시 (전 기간) — 시드만 반복 학습
# ----------------
def train_all_keys(feat_df: pd.DataFrame):
    feature_cols = CFG.FEATS
    menus = feat_df[CFG.KEY_COL].dropna().unique().tolist()
    trained_cache = {}; trained_cnt=skip_short=skip_nan=0

    for m in tqdm(menus, desc="Train per-key", unit="key"):
        dfm = feat_df[feat_df[CFG.KEY_COL]==m].sort_values("영업일자").reset_index(drop=True)
        mask = np.isfinite(dfm[feature_cols]).all(axis=1)
        dfm = dfm.loc[mask].reset_index(drop=True)
        if len(dfm) < (CFG.LOOKBACK + CFG.PREDICT):
            skip_short += 1; continue

        scaler = MinMaxScaler()
        dfm_sc = dfm.copy()
        dfm_sc[feature_cols] = scaler.fit_transform(dfm[feature_cols])

        X_all, Y_all = make_tensor_dataset(dfm_sc, feature_cols, CFG.LOOKBACK, CFG.PREDICT)
        if X_all is None:
            skip_nan += 1; continue

        epochs = CFG.EPOCHS_SPECIAL if is_special_store(get_store_name(m)) else CFG.EPOCHS
        for fam in ["NLinear","BiLSTM"]:
            for sd in CFG.SEEDS:  # ✅ 시드만 반복
                set_seed(sd)
                model = train_tensor_model(X_all, Y_all, fam, epochs=epochs)
                trained_cache[(m, fam, sd)] = {"model": model, "scaler": scaler}
                trained_cnt += 1

    tqdm.write(f"✅ 학습 완료 모델 수: {trained_cnt} | 스킵(길이): {skip_short}, 스킵(NaN): {skip_nan}")
    return trained_cache

# ----------------
# 6) 홀드아웃 평가 (tqdm 실시간 sMAPE) — family + seed 가중 앙상블
# ----------------
def evaluate_holdout_streaming(feat_df: pd.DataFrame, trained_cache: dict,
                               show_horizon_every: int = 10, zero_ignore: bool = True):
    rows = []
    Y_all, P_all = [], []
    keys = list(feat_df[CFG.KEY_COL].dropna().unique())
    bar = tqdm(total=len(keys), desc="Eval (holdout)", unit="key")

    seed_ws = get_seed_weights()  # ✅ 3시드 가중

    for idx, m in enumerate(keys, start=1):
        g = feat_df[feat_df[CFG.KEY_COL] == m].sort_values("영업일자").reset_index(drop=True)
        if len(g) < (CFG.LOOKBACK + CFG.PREDICT):
            bar.update(1); continue

        y_true = g["매출수량"].values[-CFG.PREDICT:]
        wb, wn = get_family_weights_for_key(m)
        num = np.zeros(CFG.PREDICT, dtype=float); den = 0.0
        for fam in ["NLinear", "BiLSTM"]:
            fam_w = wb if fam == "BiLSTM" else wn
            for sd, sw in zip(CFG.SEEDS, seed_ws):  # ✅ seed 가중 포함
                key = (m, fam, sd)
                if key not in trained_cache:
                    continue
                yhat = predict_next7_from_recent(
                    g, CFG.FEATS,
                    trained_cache[key]["scaler"],
                    trained_cache[key]["model"]
                )
                if (yhat is None) or (len(yhat) < CFG.PREDICT):
                    continue
                w = fam_w * sw
                num += np.array(yhat) * w; den += w
        if den == 0:
            bar.update(1); continue

        y_pred = (num / den)
        rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
        smape = smape_np(y_true, y_pred, zero_ignore=zero_ignore)
        rows.append({"key": m, "rmse": rmse, "smape": smape})

        Y_all.append(y_true)
        P_all.append(y_pred)

        # 진행바 업데이트 (현재 평균 sMAPE% + H1~H7 간이)
        smapes = [r["smape"] for r in rows if np.isfinite(r["smape"])]
        avg_smape = float(np.mean(smapes)) if len(smapes) else np.nan

        postfix = {"avg sMAPE": f"{avg_smape*100:.2f}%"}
        if (idx % show_horizon_every == 0) and len(Y_all) > 0:
            Y = np.vstack(Y_all); P = np.vstack(P_all)
            for h in range(CFG.PREDICT):
                val = smape_np(Y[:, h], P[:, h], zero_ignore=zero_ignore)
                postfix[f"H{h+1}"] = f"{val*100:.1f}%"

        bar.set_postfix(postfix); bar.update(1)

    bar.close()
    eval_df = pd.DataFrame(rows)
    if eval_df.empty:
        print("⚠️ Validation 결과 없음");
        return eval_df, None, None

    Y = np.vstack(Y_all); P = np.vstack(P_all)
    overall = {
        "rmse_overall": float(np.sqrt(np.mean((P - Y) ** 2))),
        "smape_overall": float(smape_np(Y.flatten(), P.flatten(), zero_ignore=zero_ignore)),
        "smape_overall_pct": float(smape_np(Y.flatten(), P.flatten(), zero_ignore=zero_ignore) * 100),
        "n_keys": int(eval_df.shape[0]),
    }
    per_h = []
    for h in range(CFG.PREDICT):
        val = smape_np(Y[:, h], P[:, h], zero_ignore=zero_ignore)
        per_h.append({"horizon": h + 1, "smape": float(val), "smape_pct": float(val * 100)})
    per_h_df = pd.DataFrame(per_h)

    print("===== Evaluation Summary =====")
    print(f"rmse_overall: {overall['rmse_overall']:.6f}")
    print(f"smape_overall: {overall['smape_overall']:.6f}  ({overall['smape_overall_pct']:.2f}%)")
    print(f"n_keys: {overall['n_keys']}")
    return eval_df, overall, per_h_df

# ----------------
# 7) 학습/검증 파이프라인 (train.csv)
# ----------------
def build_train_features_and_models(train_df_raw: pd.DataFrame):
    # 1) EN 표준화
    en = load_any_csv_to_en(train_df_raw)

    # 2) 스케줄/캘린더
    en = add_schedule_and_calendar(en)

    # 3) 라벨 인코딩 (train에서 fit)
    label_maps = fit_label_maps(en)
    en = apply_label_maps(en, label_maps)

    # 4) 메뉴별 판매 분위 맵 (train 기준) → 적용
    quart_map = compute_menu_quartile_map(en)
    en = apply_sales_quartile_map(en, quart_map)

    # 5) TS 피처 (요청 항목들)
    en = add_ts_features(en)

    # 6) KR 스펙 + 요청 피처만
    kr = en_to_kr_with_feats(en)

    # 7) 이상치 보간(1~99) (KR 스펙에서)
    kr = smooth_outliers_percentile_kr(kr, 0.01, 0.99)

    # 8) NaN 정리/형변환
    kr = finalize_kr_features(kr)

    # 9) 최종 FEATS 구성 (첫 열은 매출수량)
    used_feats = [c for c in REQ_FEATURES if c in kr.columns]
    CFG.FEATS = ['매출수량'] + used_feats
    print(f"✅ 최종 Feature 개수: {len(CFG.FEATS)}")
    print("Features:", CFG.FEATS)

    # 10) 학습
    trained_cache = train_all_keys(kr)

    # 11) 검증 (마지막 28 → 7)
    eval_df, overall, per_h_df = evaluate_holdout_streaming(kr, trained_cache, show_horizon_every=10)

    # 저장(옵션)
    try:
        if overall:
            pd.DataFrame([overall]).to_csv(os.path.join(BASE_PATH, "validation_overall.csv"),
                                           index=False, encoding="utf-8-sig")
        if per_h_df is not None:
            per_h_df.to_csv(os.path.join(BASE_PATH, "validation_per_horizon.csv"),
                            index=False, encoding="utf-8-sig")
        if not eval_df.empty:
            eval_df.to_csv(os.path.join(BASE_PATH, "validation_by_key.csv"),
                           index=False, encoding="utf-8-sig")
    except Exception as e:
        print("⚠️ validation 저장 실패:", e)

    return kr, trained_cache, label_maps, quart_map

# ----------------
# 8) TEST_*.csv 전체 예측 → submission(long/wide)
# ----------------
def build_test_and_predict(trained_cache: dict, label_maps: dict, quart_map: dict):
    test_files = sorted(glob.glob(TEST_GLOB))
    if not test_files:
        print("⚠️ TEST_*.csv 없음"); return pd.DataFrame()

    seed_ws = get_seed_weights()  # ✅ 3시드 가중
    all_preds = []
    for path in tqdm(test_files, desc="Predict on TEST", unit="file"):
        raw = pd.read_csv(path)
        en  = load_any_csv_to_en(raw)
        en  = add_schedule_and_calendar(en)
        en  = apply_label_maps(en, label_maps)
        en  = apply_sales_quartile_map(en, quart_map)   # train 기준 분위 사용
        en  = add_ts_features(en)

        kr  = en_to_kr_with_feats(en)
        kr  = smooth_outliers_percentile_kr(kr, 0.01, 0.99)
        kr  = finalize_kr_features(kr)

        fname = os.path.basename(path)
        mobj  = re.search(r'(TEST_\d+)', fname)
        test_prefix = mobj.group(1) if mobj else os.path.splitext(fname)[0]

        rows=[]
        for m, g in kr.groupby(CFG.KEY_COL):
            g = g.sort_values('영업일자')
            if len(g) < CFG.LOOKBACK:
                continue
            wb, wn = get_family_weights_for_key(m)   # ✅ per-key family 가중
            num = np.zeros(CFG.PREDICT, dtype=float); den = 0.0
            for fam in ["NLinear","BiLSTM"]:
                fam_w = wb if fam=="BiLSTM" else wn
                for sd, sw in zip(CFG.SEEDS, seed_ws):  # ✅ seed 가중 포함
                    key = (m, fam, sd)
                    if key not in trained_cache: continue
                    yhat = predict_next7_from_recent(
                        g, CFG.FEATS,
                        trained_cache[key]["scaler"],
                        trained_cache[key]["model"]
                    )
                    if (yhat is None) or (len(yhat) < CFG.PREDICT): continue
                    w = fam_w * sw
                    num += np.array(yhat) * w; den += w
            if den == 0:
                continue
            ens = (num/den)
            for i, val in enumerate(ens.tolist(), start=1):
                rows.append({
                    "영업일자": f"{test_prefix}+{i}일",
                    "영업장명_메뉴명": m,
                    "매출수량": float(max(0.0, val))
                })
        if rows:
            all_preds.append(pd.DataFrame(rows))

    if not all_preds:
        print("⚠️ 예측 결과 비어 있음");
        return pd.DataFrame(columns=["영업일자","영업장명_메뉴명","매출수량"])

    pred_long = pd.concat(all_preds, ignore_index=True)
    return pred_long

# ----------------
# 9) 메인
# ----------------
def main():
    # per-key family 가중 CSV 로드 (있으면 사용)
    load_family_weight_csv()

    # Train
    train_raw = pd.read_csv(TRAIN_PATH)
    feat_train_kr, trained_cache, label_maps, quart_map = build_train_features_and_models(train_raw)

    # Predict TEST
    pred_long = build_test_and_predict(trained_cache, label_maps, quart_map)

    # Save long
    out_csv = os.path.join(BASE_PATH, "submission.csv")
    try:
        pred_long.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"✅ 저장 완료: {out_csv}")
    except PermissionError:
        alt = os.path.join(BASE_PATH, f"submission_{datetime.now():%Y%m%d_%H%M%S}.csv")
        pred_long.to_csv(alt, index=False, encoding="utf-8-sig")
        out_csv = alt
        print(f"⚠️ 기본 경로 저장 실패 → 대체 경로로 저장: {alt}")

    # Save wide (sample_submission order)
    try:
        sample = pd.read_csv(SAMPLE_SUB_PATH)
        cols = [c for c in sample.columns if c != "영업일자"]

        pivot = pred_long.pivot_table(index="영업일자",
                                      columns=CFG.KEY_COL,
                                      values="매출수량",
                                      aggfunc="mean")
        pivot = pivot.reindex(index=sample["영업일자"].values, columns=cols).fillna(0.0)
        out_df = pd.concat([sample[["영업일자"]].reset_index(drop=True),
                            pivot.reset_index(drop=True)], axis=1)
        wide_csv = os.path.join(BASE_PATH, "submission_wide.csv")
        out_df.to_csv(wide_csv, index=False, encoding="utf-8-sig")
        print(f"✅ Wide 저장: {wide_csv}")
    except Exception as e:
        print(f"⚠️ Wide 변환 실패: {e}")

if __name__ == "__main__":
    main()
